In [1]:
import pandas as pd

# 1)

In [2]:
bikes_raw = pd.read_csv('../data/bike_rentals.csv')
bikes = bikes_raw.copy()

In [3]:
bikes.info()

<class 'pandas.DataFrame'>
RangeIndex: 403 entries, 0 to 402
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   rental_id      403 non-null    str    
 1   station        403 non-null    str    
 2   bike_type      403 non-null    str    
 3   helmets_text   399 non-null    str    
 4   is_member      403 non-null    str    
 5   insured        400 non-null    str    
 6   paid_online    403 non-null    str    
 7   status         403 non-null    str    
 8   coupon_code    0 non-null      float64
 9   damage_notes   0 non-null      float64
 10  late_fee_note  0 non-null      float64
 11  start_date     403 non-null    str    
 12  end_date       403 non-null    str    
 13  price          352 non-null    str    
 14  rating         288 non-null    float64
dtypes: float64(4), str(11)
memory usage: 47.4 KB


In [4]:
bikes = bikes.dropna(axis=1, how='all')
bikes.info()

<class 'pandas.DataFrame'>
RangeIndex: 403 entries, 0 to 402
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   rental_id     403 non-null    str    
 1   station       403 non-null    str    
 2   bike_type     403 non-null    str    
 3   helmets_text  399 non-null    str    
 4   is_member     403 non-null    str    
 5   insured       400 non-null    str    
 6   paid_online   403 non-null    str    
 7   status        403 non-null    str    
 8   start_date    403 non-null    str    
 9   end_date      403 non-null    str    
 10  price         352 non-null    str    
 11  rating        288 non-null    float64
dtypes: float64(1), str(11)
memory usage: 37.9 KB


In [5]:
print(bikes.shape)

(403, 12)


# 2)

In [6]:
bikes.price.head()

0      SEK 855.50
1    SEK 1,311.50
2      SEK 350.00
3      SEK 569.50
4      SEK 828.00
Name: price, dtype: str

In [7]:
price = bikes.price.str.replace('SEK', '')
print(price.head())

0       855.50
1     1,311.50
2       350.00
3       569.50
4       828.00
Name: price, dtype: str


In [8]:
price = price.str.replace(',', '')
print(price.head())

0      855.50
1     1311.50
2      350.00
3      569.50
4      828.00
Name: price, dtype: str


In [9]:
bikes.price = price.astype(float)
print(bikes.price.dtypes)
print(bikes.price.describe())
print(bikes[['rental_id', 'bike_type', 'status', 'price']].sort_values('price', ascending=False).head(3).round(2))

float64
count     352.000000
mean      847.348011
std       633.435513
min       154.500000
25%       406.000000
50%       668.250000
75%      1194.625000
max      3401.000000
Name: price, dtype: float64
    rental_id   bike_type     status   price
187    R-5282  Cargo bike  Completed  3401.0
49     R-5209  Cargo bike  Completed  2979.5
270    R-5378      E-bike  Completed  2941.0


# 3)

In [10]:
print(bikes.price.isna().sum())
print((bikes.price.isna().mean()*100).round(2))

51
12.66


In [11]:
bikes.status.value_counts()

status
Completed    334
Cancelled     47
No-show       22
Name: count, dtype: int64

In [12]:
pd.crosstab(bikes.price.isna(), bikes.status == 'Cancelled')

status,False,True
price,,
False,352,0
True,4,47


all 47 'Cancelled' have price missing (valid reason). Another 4 bikes have price missing even though they are not 'Cancelled' (accident?)

In [13]:
print(bikes.loc[(bikes.price.isna()) & (bikes.status != 'Cancelled')])

    rental_id      station  bike_type helmets_text is_member insured  \
68     R-5068  Kungsholmen  City bike     1 helmet         f       f   
108    R-5377  Kungsholmen  City bike    2 helmets         f       t   
192    R-5375     Hammarby     E-bike     1 helmet         t       f   
241    R-5231    Östermalm  City bike    2 helmets         f     NaN   

    paid_online     status  start_date    end_date  price  rating  
68            t  Completed  2026-08-07  2026-08-13    NaN     4.0  
108           f  Completed  2026-08-13  2026-08-16    NaN     5.0  
192           t  Completed  2026-03-24  2026-03-25    NaN     3.0  
241           t  Completed  2026-07-19  2026-07-20    NaN     NaN  


leaving those 4 as is. Looks legit, but no idea why price is missing.

# 4)

In [14]:
tf_columns = [col for col in bikes.columns if set(bikes[col].dropna().unique()) <= {'t', 'f'}]
print(tf_columns)

['is_member', 'insured', 'paid_online']


In [15]:
for col in tf_columns:
    print(col, bikes[col].isna().sum())
    bikes[col] = bikes[col].map({'t': True, 'f': False})

is_member 0
insured 3
paid_online 0


In [16]:
print(bikes.info())
print(bikes.head())

<class 'pandas.DataFrame'>
RangeIndex: 403 entries, 0 to 402
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   rental_id     403 non-null    str    
 1   station       403 non-null    str    
 2   bike_type     403 non-null    str    
 3   helmets_text  399 non-null    str    
 4   is_member     403 non-null    bool   
 5   insured       400 non-null    object 
 6   paid_online   403 non-null    bool   
 7   status        403 non-null    str    
 8   start_date    403 non-null    str    
 9   end_date      403 non-null    str    
 10  price         352 non-null    float64
 11  rating        288 non-null    float64
dtypes: bool(2), float64(2), object(1), str(7)
memory usage: 32.4+ KB
None
  rental_id    station  bike_type helmets_text  is_member insured  \
0    R-5020  Södermalm     E-bike     1 helmet      False    True   
1    R-5100  Södermalm     E-bike     1 helmet      False    True   
2    R-5055  Södermalm

In [17]:
for col in tf_columns:
    print(col, bikes[col].isna().sum())

is_member 0
insured 3
paid_online 0


In [21]:
print(bikes.price.where(bikes.is_member).describe())
print(bikes.price.where(bikes.is_member == False).describe())

count     104.000000
mean      734.951923
std       540.770318
min       154.500000
25%       390.375000
50%       558.000000
75%       980.250000
max      2880.000000
Name: price, dtype: float64
count     248.000000
mean      894.481855
std       663.843871
min       171.000000
25%       413.500000
50%       756.250000
75%      1271.500000
max      3401.000000
Name: price, dtype: float64


# 5)

In [23]:
print(bikes.start_date.isna().sum())
print(bikes.end_date.isna().sum())

0
0


In [25]:
print(bikes.start_date.head(3))
print(bikes.end_date.head(3))

0    2026-04-27
1    2026-09-16
2    2026-08-04
Name: start_date, dtype: str
0    2026-04-29
1    2026-09-19
2    2026-08-06
Name: end_date, dtype: str


In [26]:
for col in ['start_date', 'end_date']:
    bikes[col] = pd.to_datetime(bikes[col], errors='coerce')

In [27]:
print(bikes.start_date.isna().sum())
print(bikes.end_date.isna().sum())
print(bikes.start_date.head(3))
print(bikes.end_date.head(3))

2
0
0   2026-04-27
1   2026-09-16
2   2026-08-04
Name: start_date, dtype: datetime64[us]
0   2026-04-29
1   2026-09-19
2   2026-08-06
Name: end_date, dtype: datetime64[us]


In [28]:
print(bikes_raw.loc[bikes.start_date.isna()])

    rental_id   station bike_type helmets_text is_member insured paid_online  \
58     R-5335  Hammarby    E-bike    2 helmets         t       t           t   
292    R-5063   Central    E-bike    2 helmets         t       t           f   

        status  coupon_code  damage_notes  late_fee_note  start_date  \
58   Completed          NaN           NaN            NaN  2026-02-30   
292  Completed          NaN           NaN            NaN  2026-06-31   

       end_date       price  rating  
58   2026-05-27  SEK 702.00     3.0  
292  2026-08-09  SEK 755.50     5.0  


In [29]:
print(bikes.loc[bikes.start_date.isna()])

    rental_id   station bike_type helmets_text  is_member insured  \
58     R-5335  Hammarby    E-bike    2 helmets       True    True   
292    R-5063   Central    E-bike    2 helmets       True    True   

     paid_online     status start_date   end_date  price  rating  
58          True  Completed        NaT 2026-05-27  702.0     3.0  
292        False  Completed        NaT 2026-08-09  755.5     5.0  


In [30]:
bikes['rental_days'] = (bikes.end_date - bikes.start_date).dt.days

In [31]:
print(bikes.rental_days.describe())

count    401.000000
mean       2.955112
std        1.835206
min       -2.000000
25%        1.000000
50%        3.000000
75%        4.000000
max        7.000000
Name: rental_days, dtype: float64


problems:
1) 2 start days were not valid dates
2) minimum rental days is -2 (end date before start date)

# 6)

In [33]:
print(bikes.rating.isna().sum())

115


In [35]:
pd.crosstab(bikes.rating.isna() == False, bikes.status == 'Completed')

status,False,True
rating,,
False,69,46
True,0,288


In [36]:
bikes['has_rating'] = bikes.rating.isna() == False

In [39]:
print(bikes.has_rating.sum())

288


In [38]:
bikes.rating.describe()

count    288.000000
mean       4.225694
std        1.019547
min        1.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        5.000000
Name: rating, dtype: float64

# 7)

In [42]:
print(bikes.helmets_text.value_counts(dropna=False))

helmets_text
1 helmet          146
2 helmets         127
No helmets         59
1 child helmet     47
3 helmets          20
NaN                 4
Name: count, dtype: int64


setting NaN as 0 in helmets_count seems reasonable

In [46]:
bikes['child_helmet'] = bikes.helmets_text.str.lower().str.contains('child')
bikes['helmets_count'] = bikes.helmets_text.str.lower().str.extract(r'([\d]+)', expand=False).fillna(0).astype(int)

In [47]:
print(bikes.child_helmet.value_counts())
print(bikes.helmets_count.value_counts())

child_helmet
False    356
True      47
Name: count, dtype: int64
helmets_count
1    193
2    127
0     63
3     20
Name: count, dtype: int64


# 8)

In [49]:
print(len(bikes.station.value_counts()))
print(bikes.bike_type.value_counts())
print(bikes.status.value_counts())

20
bike_type
City bike     233
E-bike        126
Cargo bike     44
Name: count, dtype: int64
status
Completed    334
Cancelled     47
No-show       22
Name: count, dtype: int64


In [50]:
print(len(bikes.station.str.strip().str.lower().value_counts()))

5


In [51]:
bikes.station = bikes.station.str.strip().str.title()

In [52]:
print(bikes.station.value_counts())

station
Central        107
Södermalm       95
Kungsholmen     89
Östermalm       57
Hammarby        55
Name: count, dtype: int64


In [54]:
print(bikes.duplicated().sum())
print(bikes.rental_id.duplicated().sum())

3
3


In [58]:
bikes = bikes.drop_duplicates().reset_index(drop=True)

In [59]:
print(bikes.shape)
print(bikes.duplicated().sum())

(400, 16)
0


In [60]:
bikes.to_csv('../data/bikes_clean.csv', index=False)